# 第 9 章 決定木

「どの質問をすれば、もっともよくデータが分かれるか」を貪欲に選び続けて木を育てます。

対応する記事: [第 9 章 決定木（F# 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/fsharp/ch09.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch09DecisionTrees.fs"

open GrokkingMl.Ch09DecisionTrees

## 不純度

**ジニ不純度は「ランダムに 2 つ選んだとき、ラベルが食い違う確率」**、**エントロピーは「ラベルを 1 つ伝えるのに必要なビット数」** と読めます。値の範囲は違いますが、順序は一致します。

In [2]:
printfn "%-20s %8s %14s" "集合" "ジニ" "エントロピー"

for (name, ls) in [ "純粋 [1,1,1]", [ 1; 1; 1 ]
                    "偏り [1,1,1,0]", [ 1; 1; 1; 0 ]
                    "均等 [1,1,0,0]", [ 1; 1; 0; 0 ]
                    "4 クラス均等", [ 0; 1; 2; 3 ] ] do
    printfn "%-20s %8.4f %14.4f" name (giniImpurity ls) (entropy ls)

集合                  

      ジニ

        エントロピー

純粋 [1,1,1]          

  0.0000

...

  0.5000

        1.0000

4 クラス均等             

  0.7500

        2.0000

## データセット

原著と同じアプリ推薦データです。特徴量は性別（0=女性、1=男性）と年齢。**「若い人には推薦する」という規則が隠れていますが、性別は関係ありません。**

In [3]:
let points: Point list =
    [ [ 1.0; 15.0 ]; [ 0.0; 25.0 ]; [ 0.0; 32.0 ]; [ 1.0; 35.0 ]
      [ 0.0; 12.0 ]; [ 1.0; 14.0 ]; [ 1.0; 55.0 ]; [ 0.0; 40.0 ] ]

let labels = [ 1; 0; 0; 0; 1; 1; 0; 0 ]

printfn "根の不純度 ジニ %.4f / エントロピー %.4f" (giniImpurity labels) (entropy labels)

根の不純度 ジニ 

0.4688

 / エントロピー 

0.9544

## すべての分割候補を評価する

**決定木は自力で「年齢が効く、性別は関係ない」を見つけます。** 「年齢 < 20」の利得が根の不純度と一致し、この 1 回の質問で不純度が 0 になることが分かります。

In [4]:
printfn "%-8s %8s %10s" "特徴量" "閾値" "情報利得"

for split in candidateSplits points do
    let partition = applySplit points labels split

    if not (List.isEmpty partition.LeftLabels || List.isEmpty partition.RightLabels) then
        let name = if split.Feature = 0 then "性別" else "年齢"
        printfn "%-8s %8.1f %10.4f" name split.Threshold
            (informationGain giniImpurity labels partition.LeftLabels partition.RightLabels)

特徴量     

      閾値

      情報利得

性別      

     0.5

...

    37.5

    0.0938

年齢      

    47.5

    0.0402

## 木を育てる

**深さ 1、葉 2 枚の木で正解率 1.0 に達します。** モデルをそのまま出力して読めるのが決定木の強みです。

In [5]:
let tree = buildTree points labels

printfn "%A" tree
printfn "深さ %d  葉の数 %d  正解率 %.2f" (depth tree) (leafCount tree) (accuracy tree points labels)

Node ({ Feature = 1
        Threshold = 20.0 }, Leaf 1, Leaf 0)

深さ 

1

  葉の数 

2

  正解率 

1.00

## ジニとエントロピーは同じ木を作る

値の絶対値は違う（0.4688 と 0.9544）のに、**順序が同じなので選ばれる分割も同じ** になります。

In [6]:
let byGini = buildTreeWith giniImpurity 5 1 points labels
let byEntropy = buildTreeWith entropy 5 1 points labels
printfn "同じ木か: %b" (byGini = byEntropy)

同じ木か: 

true

## 試してみる: 木の成長を止める

最大深さと最小サンプル数は、**第 4 章の正則化と同じ役割** です。木は放っておくと訓練データを丸暗記するまで育ちます。

In [7]:
for maxDepth in [ 0; 1; 3 ] do
    let t = buildTreeWith giniImpurity maxDepth 1 points labels
    printfn "maxDepth=%d  深さ %d  葉 %d  正解率 %.2f" maxDepth (depth t) (leafCount t)
        (accuracy t points labels)

maxDepth=

0

  深さ 

0

  葉 

...

1

  葉 

2

  正解率 

1.00